INSTALACIONES (SOLO UNA VEZ EN COLAB)

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow
!pip install torch torchvision torchaudio transformers
!pip install nltk spacy tqdm joblib
!python -m spacy download es_core_news_sm

IMPORTS

In [ ]:
# Procesamiento de datos
import pandas as pd
import numpy as np
import re
import string
import os

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y ML clásico
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, roc_auc_score)

# Deep Learning (Keras)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Transformers / BERT
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertModel

# NLP
import nltk
from nltk.corpus import stopwords
import spacy

# Utilidades
from tqdm import tqdm
from collections import Counter
import joblib

CONFIGURACIÓN BASE (STOPWORDS, SPACY, DISPOSITIVOS)

In [ ]:
# Descargar stopwords solo una vez
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))

# Cargar modelo de spaCy
nlp = spacy.load("es_core_news_sm", disable=["ner", "tagger"])

# Dispositivo para BERT (CPU forzado como en tu código original)
device = torch.device('cpu')

In [ ]:
import os
import pandas as pd

# 1) Montar Google Drive (solo en Colab)
from google.colab import drive
drive.mount('/content/drive')


CARGA DE DATOS (AJUSTA RUTAS A TU DRIVE)

In [ ]:
# 2) Ruta a la carpeta donde están tus archivos en Drive
#    Ejemplo: si tus CSV están en "Mi unidad/Proyecto/data/"
DATA_DIR = os.environ.get("FAKENEWS_DATA_DIR", "/content/drive/MyDrive/modelo_3.0")  # Ajusta la ruta

# 3) Leer archivos
df = pd.read_csv(os.path.join(DATA_DIR, "fake_news_final.csv"))
df_diccionario = pd.read_csv(os.path.join(DATA_DIR, "diccionario.csv"))

print("Info del df principal:")
df.info()

df_metadata = df.copy()
df_heuristica = df.copy()


FUNCIONES AUXILIARES DE TEXTO

In [ ]:
def limpiar_basico(texto):
    """Minúsculas + elimina puntuación básica."""
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    for c in string.punctuation:
        texto = texto.replace(c, "")
    return texto

def limpiar_texto(texto):
    """Limpieza usada para BERT."""
    texto = str(texto).lower()
    texto = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

DICCIONARIO DE EMOCIONES

In [ ]:
def construir_diccionario_emocional(df_diccionario):
    """
    Asume un df con columna 'palabra' y columnas binarias de emociones.
    Si tus columnas son distintas, ajusta aquí.
    """
    emociones = df_diccionario.columns[1:]  # todas menos 'palabra'
    diccionario_emocional = {}
    for _, fila in df_diccionario.iterrows():
        palabra = fila['palabra']
        emociones_asociadas = [emocion for emocion in emociones if fila[emocion] == 1]
        if emociones_asociadas:
            diccionario_emocional[palabra] = emociones_asociadas
    return diccionario_emocional

def calcular_emotividad(texto, diccionario):
    palabras = re.findall(r'\b\w+\b', str(texto).lower())
    total = len(palabras)
    emocionales = sum(1 for palabra in palabras if palabra in diccionario)
    return emocionales / total if total > 0 else 0

def emocion_predominante(texto, diccionario):
    palabras = re.findall(r'\b\w+\b', str(texto).lower())
    emociones = []
    for palabra in palabras:
        if palabra in diccionario:
            emociones.extend(diccionario[palabra])
    if emociones:
        return Counter(emociones).most_common(1)[0][0]
    else:
        return 'ninguna'

def aplicar_emotividad(df_heuristica, df_diccionario):
    diccionario_emocional = construir_diccionario_emocional(df_diccionario)
    df_heuristica['score_emotividad'] = df_heuristica['contenido'].apply(
        lambda x: calcular_emotividad(x, diccionario_emocional)
    )
    df_heuristica['emocion_predominante'] = df_heuristica['contenido'].apply(
        lambda x: emocion_predominante(x, diccionario_emocional)
    )
    return df_heuristica[['score_emotividad']]

HEURÍSTICAS LINGÜÍSTICAS

In [ ]:
verbos_modales = ['debería', 'podría', 'tendría', 'habría', 'puede', 'pueden',
                  'debe', 'deben', 'sería']
terminos_generalizadores = ['siempre', 'nunca', 'todos', 'nadie', 'jamás',
                            'ninguno', 'cualquiera']
verbos_opinion = ['creo', 'considero', 'opino', 'pienso', 'me parece',
                  'supongo', 'siento', 'estimo']
lexico_polarizado = ['horrible', 'excelente', 'terrible', 'maravilloso',
                     'desastroso', 'perfecto', 'abominable', 'magnífico']

def frecuencia_heuristica(texto, palabras_clave):
    texto_limpio = limpiar_basico(texto)
    palabras = texto_limpio.split()
    return sum(1 for palabra in palabras if palabra in palabras_clave)

def diversidad_lexica(texto):
    texto_limpio = limpiar_basico(texto)
    palabras = [p for p in texto_limpio.split() if p.isalpha() and p not in stop_words]
    if len(palabras) == 0:
        return 0
    return len(set(palabras)) / len(palabras)

def aplicar_heuristicas_linguisticas(df_heuristica):
    df_heuristica['incertidumbre'] = df_heuristica['contenido'].apply(
        lambda x: frecuencia_heuristica(x, verbos_modales + terminos_generalizadores)
    )
    df_heuristica['subjetividad'] = df_heuristica['contenido'].apply(
        lambda x: frecuencia_heuristica(x, verbos_opinion + lexico_polarizado)
    )
    df_heuristica['diversidad_lexica'] = df_heuristica['contenido'].apply(diversidad_lexica)
    return df_heuristica[['incertidumbre', 'subjetividad', 'diversidad_lexica']]


PATRONES SINTÁCTICOS (SPACY)

In [ ]:
def aplicar_patrones_sintacticos(df_heuristica):
    resultados = []
    for doc in nlp.pipe(df_heuristica['contenido'], batch_size=32):
        oraciones = list(doc.sents)
        total_palabras = len([token.text for token in doc if token.is_alpha])
        total_palabras_mayus = len([token.text for token in doc if token.is_upper])
        total_exclamaciones = doc.text.count('!')
        frases_cortas = sum(1 for s in oraciones if len(s.text.split()) <= 7)
        porcentaje_mayus = total_palabras_mayus / total_palabras if total_palabras > 0 else 0
        frases_cortas_porcentaje = frases_cortas / len(oraciones) if oraciones else 0
        resultados.append({
            "porcentaje_mayusculas": porcentaje_mayus,
            "exclamaciones_totales": total_exclamaciones,
            "frases_cortas": frases_cortas,
            "frases_cortas_%": frases_cortas_porcentaje
        })
    df_patrones = pd.DataFrame(resultados, index=df_heuristica.index)
    return df_patrones


RELACIÓN TÍTULO–CUERPO (JACCARD)

In [ ]:
def consistencia_jaccard(titulo, contenido):
    titulo = limpiar_basico(titulo)
    contenido = limpiar_basico(contenido)
    set_titulo = set([p for p in titulo.split() if p not in stop_words])
    set_contenido = set([p for p in contenido.split() if p not in stop_words])
    if len(set_titulo) == 0 or len(set_contenido) == 0:
        return 0
    interseccion = set_titulo.intersection(set_contenido)
    union = set_titulo.union(set_contenido)
    return len(interseccion) / len(union)

def aplicar_relacion_titulo_cuerpo(df_heuristica):
    df_heuristica['consistencia_titulo_cuerpo'] = df_heuristica.apply(
        lambda row: consistencia_jaccard(row['titulo'], row['contenido']), axis=1)
    return df_heuristica[['consistencia_titulo_cuerpo']]

EMBEDDING HEURÍSTICO (CONCATENANDO TODO LO ANTERIOR)

In [ ]:
def generar_embedding_heuristico(df, df_diccionario):
    emotividad = aplicar_emotividad(df.copy(), df_diccionario)
    heuristicas = aplicar_heuristicas_linguisticas(df.copy())
    patrones = aplicar_patrones_sintacticos(df.copy())
    relacion = aplicar_relacion_titulo_cuerpo(df.copy())

    heuristico_df = pd.concat([emotividad, heuristicas, patrones, relacion], axis=1)
    heuristico_df = heuristico_df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return heuristico_df.to_numpy()

FEATURE ENGINEERING METADATA (SIN ESCALAR)

In [ ]:
# Columnas numéricas que vamos a usar en el scaler
COL_NUM = [
    # Puntuación / estilo
    'exclamaciones',
    'interrogaciones',
    'palabras_sensacionalistas',
    'noticias_autor',
    'relacion_titulo_contenido',

    # Longitud categorizada
    'es_muy_corta',
    'es_corta',
    'es_media',
    'es_larga',
    'es_muy_larga',

    # Longitud del título categorizada
    'titulo_corto',
    'titulo_largo'
]


def feature_engineering(df):
    # -------- 1) Longitudes brutas en palabras --------
    long_contenido = df['contenido'].apply(lambda x: len(str(x).split()))
    long_titulo = df['titulo'].apply(lambda x: len(str(x).split()))

    # Las guardamos por si quieres mirarlas en EDA, pero NO van al modelo
    df['longitud_bruta'] = long_contenido
    df['longitud_titulo_bruta'] = long_titulo

    # -------- 2) Flags de longitud del contenido --------
    df['es_muy_corta']  = (long_contenido < 80).astype(int)
    df['es_corta']      = ((long_contenido >= 80) & (long_contenido < 200)).astype(int)
    df['es_media']      = ((long_contenido >= 200) & (long_contenido < 800)).astype(int)
    df['es_larga']      = ((long_contenido >= 800) & (long_contenido < 2000)).astype(int)
    df['es_muy_larga']  = (long_contenido >= 2000).astype(int)

    # -------- 3) Flags de longitud del título --------
    df['titulo_corto']  = (long_titulo < 6).astype(int)
    df['titulo_largo']  = (long_titulo > 15).astype(int)

    # -------- 4) Puntuación --------
    df['exclamaciones'] = df['contenido'].apply(lambda x: str(x).count('!'))
    df['interrogaciones'] = df['contenido'].apply(lambda x: str(x).count('?'))

    # -------- 5) Palabras sensacionalistas --------
    sensacionalistas = ['increíble', 'urgente', 'impactante', 'viral', 'escándalo']
    df['palabras_sensacionalistas'] = df['contenido'].apply(
        lambda x: sum(str(x).lower().count(p) for p in sensacionalistas)
    )

    # -------- 6) Fecha -> componentes --------
    df['fecha'] = pd.to_datetime(df['fecha'], format='%d-%m-%Y', errors='coerce')
    df['dia_semana'] = df['fecha'].dt.day_name()
    df['mes'] = df['fecha'].dt.month

    # Frecuencia por autor
    df['noticias_autor'] = df['autor'].map(df['autor'].value_counts())

    # -------- 7) Relación título / contenido (ratio acotada) --------
    ratio = long_titulo / np.maximum(long_contenido, 1)
    ratio_clipped = np.clip(ratio, 0, 0.5)
    df['relacion_titulo_contenido'] = ratio_clipped

    # -------- 8) Agrupar autores poco frecuentes en 'Otro' --------
    autor_freq = df['autor'].value_counts()
    df['autor'] = df['autor'].apply(
        lambda x: x if x in autor_freq[autor_freq > 20].index else 'Otro'
    )

    # -------- 9) Variables cíclicas de mes y día --------
    df['mes'] = df['mes'].fillna(0)
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

    dias = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
            'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    df['dia_semana_num'] = df['dia_semana'].map(dias).fillna(0).astype(int)
    df['dia_sin'] = np.sin(2 * np.pi * df['dia_semana_num'] / 7)
    df['dia_cos'] = np.cos(2 * np.pi * df['dia_semana_num'] / 7)

    # -------- 10) Limpiar columnas numéricas --------
    df[COL_NUM] = df[COL_NUM].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df


EMBEDDINGS BERT

In [ ]:
class TextDataset(Dataset):
    def __init__(self, textos):
        self.textos = textos

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        return self.textos[idx]

def generar_embedding_bert(df, tokenizer, model, columna_texto='contenido_limpio', batch_size=32):
    model.to(device)
    model.eval()

    dataset = TextDataset(df[columna_texto].tolist())
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    embeddings = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generando embeddings BERT (CPU)"):
            encoded = tokenizer(
                batch,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512
            )
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS]

            embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, dim=0).numpy()

GENERACIÓN DE EMBEDDINGS (TRAIN)

Ingeniería de metadata

In [ ]:
df_metadata = df.copy()
df_metadata = feature_engineering(df_metadata)

print(df_metadata[COL_NUM].head())  # opcional: comprobar que existen las columnas


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# numéricas
X_num = df_metadata[COL_NUM].values
scaler_metadata = StandardScaler()
X_num_scaled = scaler_metadata.fit_transform(X_num)

# cíclicas
X_ciclicas = df_metadata[['mes_sin', 'mes_cos', 'dia_sin', 'dia_cos']].values

# OneHot autor
ohe_autor = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_autor = ohe_autor.fit_transform(df_metadata[['autor']])

# metadata final
X_metadata = np.concatenate([X_num_scaled, X_ciclicas, X_autor], axis=1)

print("X_metadata shape:", X_metadata.shape)


Embeddings heurísticos

In [ ]:
print("➡️ Embeddings heurísticos...")
X_heuristico = generar_embedding_heuristico(df_heuristica, df_diccionario)
scaler_heuristic = StandardScaler()
X_heuristico_scaled = scaler_heuristic.fit_transform(X_heuristico)


Embeddings BERT

In [ ]:
print("➡️ Embeddings BERT...")
df['contenido_limpio'] = df['contenido'].astype(str).apply(limpiar_texto)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
model_bert = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
X_bert = generar_embedding_bert(df, tokenizer, model_bert)
scaler_bert = StandardScaler()
X_bert_scaled = scaler_bert.fit_transform(X_bert)

print("Shapes:")
print("X_bert_scaled:", X_bert_scaled.shape)
print("X_metadata  :", X_metadata.shape)
print("X_heuristic :", X_heuristico_scaled.shape)

# Concatenar todo
X_final = np.concatenate([X_bert_scaled, X_metadata, X_heuristico_scaled], axis=1)
y = df['label'].values

print("X_final shape:", X_final.shape)

GUARDAR TRANSFORMADORES Y FEATURES

In [ ]:
# Guardar features finales
np.save(os.path.join(DATA_DIR, "X_finalv3.npy"), X_final)

# Guardar scalers
joblib.dump(scaler_metadata, os.path.join(DATA_DIR, "scaler_metadata.pkl"))
joblib.dump(scaler_bert,     os.path.join(DATA_DIR, "scaler_bert.pkl"))
joblib.dump(scaler_heuristic,os.path.join(DATA_DIR, "scaler_heuristic.pkl"))

# Guardar OneHotEncoder
joblib.dump(ohe_autor,       os.path.join(DATA_DIR, "encoder_autor.pkl"))





PREPARAR DATOS PARA ENTRENAR EL MODELO

In [ ]:
from sklearn.model_selection import train_test_split

# Asumiendo que ya tienes:
# X_bert_scaled      -> (n_samples, 768)
# X_metadata         -> (n_samples, n_meta)
# X_heuristico_scaled -> (n_samples, n_heur)
# y                  -> (n_samples,)

# PRIMERA división: Train vs (Val+Test)
X_bert_train, X_bert_temp, \
X_metadata_train, X_metadata_temp, \
X_heuristico_train, X_heuristico_temp, \
y_train, y_temp = train_test_split(
    X_bert_scaled,
    X_metadata,
    X_heuristico_scaled,
    y,
    test_size=0.30,          # 30% para val+test
    random_state=42,
    stratify=y
)

# SEGUNDA división: Val vs Test (a partir del 30%)
X_bert_val, X_bert_test, \
X_metadata_val, X_metadata_test, \
X_heuristico_val, X_heuristico_test, \
y_val, y_test = train_test_split(
    X_bert_temp,
    X_metadata_temp,
    X_heuristico_temp,
    y_temp,
    test_size=0.50,          # 15% val, 15% test en total
    random_state=42,
    stratify=y_temp
)

print("Train shapes:",
      X_bert_train.shape, X_metadata_train.shape, X_heuristico_train.shape, len(y_train))
print("Val shapes:",
      X_bert_val.shape, X_metadata_val.shape, X_heuristico_val.shape, len(y_val))
print("Test shapes:",
      X_bert_test.shape, X_metadata_test.shape, X_heuristico_test.shape, len(y_test))


DEFINIR MODELO MULTIRAMA (BERT / METADATA / HEURÍSTICO)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, BatchNormalization, Lambda
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.metrics import AUC

# ============================
# 1) Entradas
# ============================

input_bert = Input(shape=(X_bert_scaled.shape[1],), name="input_bert")
input_meta = Input(shape=(X_metadata.shape[1],), name="input_meta")
input_heur = Input(shape=(X_heuristico_scaled.shape[1],), name="input_heur")


# ============================
# 2) Rama BERT (reducida + regularizada)
# ============================

bert_branch = Dense(256, activation='relu')(input_bert)
bert_branch = BatchNormalization()(bert_branch)
bert_branch = Dropout(0.5)(bert_branch)

bert_branch = Dense(128, activation='relu')(bert_branch)
bert_branch = Dropout(0.5)(bert_branch)

# IDEA:
# BERT se reduce para que no domine al resto.


# ============================
# 3) Rama Metadata (aumentada)
# ============================

meta_branch = Dense(256, activation='relu')(input_meta)
meta_branch = BatchNormalization()(meta_branch)
meta_branch = Dropout(0.3)(meta_branch)

meta_branch = Dense(128, activation='relu')(meta_branch)
meta_branch = Dropout(0.3)(meta_branch)

# IDEA:
# Subimos capacidad para que metadata tenga poder real.


# ============================
# 4) Rama Heurística (potenciada)
# ============================

heur_branch = Dense(64, activation='relu')(input_heur)
heur_branch = BatchNormalization()(heur_branch)
heur_branch = Dropout(0.3)(heur_branch)

heur_branch = Dense(32, activation='relu')(heur_branch)
heur_branch = Dropout(0.3)(heur_branch)

# IDEA:
# Antes tenía muy poca capacidad; ahora aporta de verdad.


# ============================
# 5) Ponderación opcional
# ============================
# Aquí literalmente le damos MÁS peso a metadata y heurística que a BERT.

meta_boost = Lambda(lambda x: x * 1.3)(meta_branch)
heur_boost = Lambda(lambda x: x * 1.6)(heur_branch)

# Si después quieres algo más agresivo, subimos estos números.


# ============================
# 6) Fusión
# ============================

fusion = Concatenate()([bert_branch, meta_boost, heur_boost])

fusion = Dense(256, activation='relu')(fusion)
fusion = BatchNormalization()(fusion)
fusion = Dropout(0.4)(fusion)

fusion = Dense(128, activation='relu')(fusion)
fusion = Dropout(0.4)(fusion)

output = Dense(1, activation='sigmoid')(fusion)


# ============================
# 7) Compilar modelo
# ============================

model_v2 = Model(
    inputs=[input_bert, input_meta, input_heur],
    outputs=output
)

model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        AUC(name="auc")
    ]
)

model_v2.summary()


CALLBACKS Y ENTRENAMIENTO

In [ ]:
import os
import joblib
import numpy as np

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

PATIENCE = 10

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=DATA_DIR + 'mejor_modelo_multirama_mejorado.h5',
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

history = model_v2.fit(
    [X_bert_train, X_metadata_train, X_heuristico_train],
    y_train,
    validation_data=([X_bert_val, X_metadata_val, X_heuristico_val], y_val),
    epochs=40,
    batch_size=16,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=6,
            monitor="val_loss",
            restore_best_weights=True
        )
    ],
    verbose=1
)

GRÁFICAS DE ENTRENAMIENTO

In [ ]:
# Guardar modelo Keras
model_v2.save(os.path.join(DATA_DIR, "mejor_modelo_multirama_mejorado.h5"))

In [ ]:
plt.figure(figsize=(16, 4))

# Loss
plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Pérdida')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 3, 2)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# AUC
plt.subplot(1, 3, 3)
plt.plot(history.history['auc'], label='Train')
plt.plot(history.history['val_auc'], label='Val')
plt.title('AUC')
plt.xlabel('Épocas')
plt.ylabel('AUC')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

# ===== Predicciones en TEST para model_v2 (multi-input) =====

# OJO: aquí usamos las tres entradas del modelo
y_pred_prob = model_v2.predict([X_bert_test, X_metadata_test, X_heuristico_test]).ravel()

# Umbral (por ahora 0.5, puedes moverlo luego si quieres)
umbral = 0.5
y_pred = (y_pred_prob > umbral).astype(int)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión (Test) - model_v2')
plt.show()

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
auc_value = roc_auc_score(y_test, y_pred_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_value:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC (Test) - model_v2')
plt.legend()
plt.grid(True)
plt.show()

print(f"AUC (Test): {auc_value:.4f}")
print(f"Umbral usado: {umbral}")


In [ ]:
def ingresar_y_clasificar_noticia(
    titulo: str,
    autor: str,
    fecha: str,
    contenido: str,
    umbral: float = 0.5,
    mostrar_detalle: bool = True
):
    """
    Clasifica una noticia como Verdadera / Falsa usando model_v2
    con las tres ramas: BERT + metadata + heurística.

    Parámetros
    ----------
    titulo : str
    autor : str
    fecha : str  (formato 'dd-mm-YYYY' idealmente)
    contenido : str
    umbral : float, opcional (default 0.5)
        Si prob >= umbral -> 'Verdadera', si no -> 'Falsa'.
    mostrar_detalle : bool
        Si True, imprime explicación con factores heurísticos.
    """
    print("📰 Analizando noticia...\n")

    # ------------------------------
    # 1) DataFrame de una fila
    # ------------------------------
    df_noticia = pd.DataFrame({
        "titulo": [titulo],
        "autor": [autor],
        "fecha": [fecha],
        "contenido": [contenido]
    })

    # ============================
    # 2) HEURÍSTICAS
    # ============================
    X_heur = generar_embedding_heuristico(df_noticia.copy(), df_diccionario)
    X_heur = scaler_heuristic.transform(X_heur)

    # ============================
    # 3) METADATA
    # ============================
    df_meta = feature_engineering(df_noticia.copy())

    # mismas columnas numéricas que en entrenamiento
    X_num = df_meta[COL_NUM].values
    X_num_scaled = scaler_metadata.transform(X_num)

    # variables cíclicas
    X_ciclicas = df_meta[['mes_sin', 'mes_cos', 'dia_sin', 'dia_cos']].values

    # autor OHE (entrenado previamente)
    X_autor = ohe_autor.transform(df_meta[['autor']])

    # metadata final
    X_metadata = np.concatenate([X_num_scaled, X_ciclicas, X_autor], axis=1)

    # ============================
    # 4) BERT
    # ============================
    df_noticia['contenido_limpio'] = df_noticia['contenido'].astype(str).apply(limpiar_texto)

    inputs = tokenizer(
        df_noticia['contenido_limpio'].iloc[0],
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512
    )

    # usamos el mismo device que en el entrenamiento de BERT
    global device
    if 'device' not in globals():
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model_bert.to(device)

    for k in inputs:
        inputs[k] = inputs[k].to(device)

    with torch.no_grad():
        outputs = model_bert(**inputs)
        # [CLS]
        X_bert = outputs.last_hidden_state[:, 0, :].cpu().numpy().reshape(1, -1)

    X_bert = scaler_bert.transform(X_bert)

    # ============================
    # 5) PREDICCIÓN CON model_v2
    # ============================
    prob = float(model_v2.predict([X_bert, X_metadata, X_heur], verbose=0)[0][0])
    clasificacion = "Verdadera" if prob >= umbral else "Falsa"

    print("📊 Resultados del modelo v2")
    print(f"- Probabilidad de ser verdadera: {prob*100:.2f}%")
    print(f"- Umbral usado: {umbral:.2f}")
    print(f"- Clasificación: {clasificacion}")

    if not mostrar_detalle:
        return clasificacion, prob

    # ============================
    # 6) EXPLICACIÓN RÁPIDA
    # ============================
    print("\n🔎 Factores heurísticos (explicación rápida):")
    titulo_ = df_noticia['titulo'].iloc[0]
    contenido_ = df_noticia['contenido'].iloc[0]
    autor_ = df_noticia['autor'].iloc[0]

    # Coincidencia título–cuerpo
    coincidencias = len(
        set(limpiar_texto(titulo_).split()).intersection(
            set(limpiar_texto(contenido_).split())
        )
    )
    print(f"- Coincidencias título-contenido: {coincidencias} palabras clave")
    print(f"- Autor: {autor_}")

    longitud = len(contenido_.split())
    print(f"- Longitud del contenido: {longitud} palabras")

    exclamaciones = contenido_.count("!")
    print(f"- Signos de exclamación: {exclamaciones}")

    dic_emocional = construir_diccionario_emocional(df_diccionario)
    emotividad = calcular_emotividad(contenido_, dic_emocional)
    emocion = emocion_predominante(contenido_, dic_emocional)
    print(f"- Score de emotividad: {emotividad:.3f} (emoción predominante: {emocion})")

    return clasificacion, prob


In [ ]:
ingresar_y_clasificar_noticia(
    titulo="La OMS aprueba una segunda vacuna contra el dengue para uso global",
    autor="BBC News",
    fecha="03-10-2023",
    contenido="""
La Organización Mundial de la Salud (OMS) anunció la aprobación de una
segunda vacuna contra el dengue destinada a reforzar la respuesta global
frente al aumento de casos en regiones tropicales.

La vacuna, desarrollada por el laboratorio japonés Takeda, obtuvo la
recomendación del Grupo Estratégico Asesor de Expertos en Inmunización
de la OMS tras analizar los datos de seguridad y eficacia recopilados
durante los últimos años.

Según la OMS, el dengue continúa expandiéndose debido al aumento de
temperaturas y la urbanización acelerada, por lo que contar con nuevas
herramientas de prevención es fundamental para los sistemas de salud pública.

El organismo señaló que la vacuna podrá ser utilizada en países donde el
virus es endémico y que se espera su distribución progresiva durante 2024.
"""
)
